# Ingest – Open-Meteo historical weather → raw CSV

Daily weather for every city in `config/cities.csv`, from `START_DATE` (2023-01-01) until yesterday.
Settings are in `config/settings.py`.

```
raw_data/weather/deltadate=<YYYY-MM-DD>/<City>.csv     daily weather
raw_data/cities/deltadate=<YYYY-MM-DD>/cities.csv     country, population ... (geocoding API, once per city)
```

* **deltadate** – everything up to `INITIAL_DELTADATE` (2026-10-06) is the initial load and has that deltadate,
  every later day has its own date as deltadate. A new city's history starts
  in `START_DATE`, so its weather and its row in `cities.csv` start in the `INITIAL_DELTADATE` folder.
* Each run continues after the last deltadate already saved for a city, so re-running only fetches new days.
  Days the archive doesn't have yet (empty rows) are skipped and come with a later run.
* `STORAGE` in `.env`: `local` = `raw_data/` in the project, `s3` = bucket on Neon Object Storage (same keys).

## Setup

In [ ]:
import sys
import time
from datetime import date, timedelta
from pathlib import Path

import boto3
import openmeteo_requests
import pandas as pd
import requests_cache
from botocore.config import Config
from retry_requests import retry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from config.settings import (ARCHIVE_URL, CITY_COLUMNS, DAILY_VARIABLES, GEOCODING_URL, INITIAL_DELTADATE,
                             RAW_DIR, S3_ACCESS_KEY_ID, S3_BUCKET, S3_ENDPOINT_URL, S3_REGION,
                             S3_SECRET_ACCESS_KEY, START_DATE, STORAGE)

# Setup the Open-Meteo API client with cache and retry on error (as in the API docs)
cache_session = requests_cache.CachedSession(str(ROOT / ".cache" / "openmeteo"), expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

raw = ROOT / RAW_DIR
if STORAGE == "s3":
    s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT_URL, region_name=S3_REGION, aws_access_key_id=S3_ACCESS_KEY_ID,
                      aws_secret_access_key=S3_SECRET_ACCESS_KEY, config=Config(s3={"addressing_style": "path"}))


def list_keys(prefix):
    """Keys of the raw CSV files under prefix, e.g. 'weather/' -> ['weather/deltadate=2026-10-06/Atlanta.csv', ...]"""
    if STORAGE == "s3":
        pages = s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=prefix)
        return sorted(o["Key"] for page in pages for o in page.get("Contents", []) if o["Key"].endswith(".csv"))
    return sorted(p.relative_to(raw).as_posix() for p in (raw / prefix).rglob("*.csv"))


def read_csv(key):
    if STORAGE == "s3":
        return pd.read_csv(s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"])
    return pd.read_csv(raw / key)


def save(df, key):
    if STORAGE == "s3":
        s3.put_object(Bucket=S3_BUCKET, Key=key, Body=df.to_csv(index=False).encode("utf-8"), ContentType="text/csv")
    else:
        path = raw / key
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path, index=False)


def deltadate_of(key):
    return key.split("/")[1].split("=")[1]          # weather/deltadate=2026-10-06/Atlanta.csv -> 2026-10-06


end_date = date.today() - timedelta(days=1)
print(f"{STORAGE}: {S3_BUCKET if STORAGE == 's3' else raw} | {START_DATE} .. {end_date} | initial deltadate {INITIAL_DELTADATE}")

## Cities

In [ ]:
cities = pd.read_csv(ROOT / "config" / "cities.csv")
cities["file"] = cities["city"].str.replace(" ", "_") + ".csv"
cities

## Cities – geocoding
Country, population ... from the Open-Meteo geocoding API (free, not counted in the weather quota).
Several places share a name, so the result with the same name closest to the configured coordinates is used.
Only cities that are in none of the `raw_data/cities/deltadate=*/cities.csv` files are requested; they are saved
together in `cities/deltadate=<INITIAL_DELTADATE>/cities.csv`, where their weather history starts as well.

In [ ]:
def geocode(city):
    results = retry_session.get(GEOCODING_URL, params={"name": city["city"], "count": 10, "language": "en"}).json()
    results = pd.DataFrame(results.get("results", []))
    if results.empty:
        return None
    same_name = results[results["name"].str.lower() == city["city"].lower()]
    if not same_name.empty:
        results = same_name
    distance = ((results["latitude"] - city["latitude"]) ** 2 + (results["longitude"] - city["longitude"]) ** 2) ** 0.5
    match = results.loc[[distance.idxmin()]]
    match.insert(0, "city", city["city"])
    return match


city_files = list_keys("cities/")
geocoded = set(pd.concat([read_csv(k) for k in city_files])["city"]) if city_files else set()

found = []
for _, city in cities[~cities["city"].isin(geocoded)].iterrows():
    match = geocode(city)
    if match is None:
        print(f"{city['city']}: not found")
        continue
    found.append(match)
    print(f"{city['city']}: {match.iloc[0]['name']}, {match.iloc[0]['country']}")

if found:
    key = f"cities/deltadate={INITIAL_DELTADATE}/cities.csv"
    new = pd.concat(found).reindex(columns=CITY_COLUMNS).assign(deltadate=INITIAL_DELTADATE)
    if key in city_files:                             # cities added later: append to the file
        new = pd.concat([read_csv(key), new])
    save(new, key)
else:
    print("all cities geocoded")

pd.concat([read_csv(k) for k in list_keys("cities/")])

## Weather – functions
* `last_deltadate` – newest deltadate already saved for a city
* `fetch` – request + processing as in the API docs, dates are the city's local dates

In [ ]:
def last_deltadate(city, keys):
    days = [deltadate_of(k) for k in keys if k.endswith("/" + city["file"])]
    return max((date.fromisoformat(d) for d in days), default=None)


def fetch(city, start, end):
    params = {
        "latitude": city["latitude"],
        "longitude": city["longitude"],
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": DAILY_VARIABLES,
        "timezone": "auto",              # daily values over the city's local day
    }
    for attempt in range(3):
        try:
            response = openmeteo.weather_api(ARCHIVE_URL, params=params)[0]
            break
        except Exception as e:           # free tier: 600 calls/min -> wait and try again
            if "limit" not in str(e).lower() or attempt == 2:
                raise
            print(f"  {e} -> waiting 60 s")
            time.sleep(60)

    daily = response.Daily()
    offset = response.UtcOffsetSeconds()
    data = {"date": pd.date_range(
        start=pd.to_datetime(daily.Time() + offset, unit="s"),
        end=pd.to_datetime(daily.TimeEnd() + offset, unit="s"),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left",
    ).date}
    for i, name in enumerate(DAILY_VARIABLES):
        data[name] = daily.Variables(i).ValuesAsNumpy()

    df = pd.DataFrame(data).dropna(subset=DAILY_VARIABLES, how="all")   # days not in the archive yet
    df.insert(0, "city", city["city"])
    df.insert(1, "latitude", response.Latitude())        # grid cell the API used
    df.insert(2, "longitude", response.Longitude())
    df.insert(3, "elevation", response.Elevation())
    df.insert(4, "timezone", response.Timezone().decode())
    df.insert(5, "deltadate", df["date"].clip(lower=date.fromisoformat(INITIAL_DELTADATE)))
    return df

## Load weather

In [ ]:
weather_keys = list_keys("weather/")
log = []
for _, city in cities.iterrows():
    last = last_deltadate(city, weather_keys)
    start = last + timedelta(days=1) if last else date.fromisoformat(START_DATE)
    if start > end_date:
        print(f"{city['city']}: up to date")
        continue

    df = fetch(city, start, end_date)
    for deltadate, part in df.groupby("deltadate"):
        save(part, f"weather/deltadate={deltadate}/{city['file']}")
        log.append({"city": city["city"], "deltadate": deltadate,
                    "from": part["date"].min(), "to": part["date"].max(), "rows": len(part)})
    print(f"{city['city']}: {start} .. {end_date} -> {len(df)} days")

pd.DataFrame(log)

## Saved files

In [ ]:
files = pd.DataFrame([{"deltadate": deltadate_of(k), "city": k.rsplit("/", 1)[1][:-4]}
                      for k in list_keys("weather/")])
files.groupby("deltadate")["city"].agg(["count", ", ".join]) if len(files) else files